In [4]:
!pip install adapters -q

In [5]:
import pandas as pd
from transformers import TrainingArguments, AutoModelForSequenceClassification, EarlyStoppingCallback, AutoTokenizer, set_seed, TrainerCallback
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint
import adapters
from adapters import DoubleSeqBnConfig, AdapterTrainer
from transformers import DataCollatorWithPadding
!pip install iterative-stratification
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


Loading the dataset

In [6]:
drive.mount('/content/drive', force_remount=True)

SPLIT_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv"
MLB_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/scotbess_mlb.joblib"

output_dir = "/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_LC/SCOTBESS_MODERNBERT_HOULSBY_LC"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


In [7]:
scotbess_df = pd.read_csv(SPLIT_PATH)

In [8]:
scotbess_df_train = scotbess_df[scotbess_df["split"] == "train"].reset_index(drop=True)
scotbess_df_val = scotbess_df[scotbess_df["split"] == "validation"].reset_index(drop=True)
scotbess_df_test = scotbess_df[scotbess_df["split"] == "test"].reset_index(drop=True)

print("Train shape:", scotbess_df_train.shape)
print("Validation shape:", scotbess_df_val.shape)
print("Test shape:", scotbess_df_test.shape)

Train shape: (1340, 7)
Validation shape: (165, 7)
Test shape: (170, 7)


In [9]:
mlb = joblib.load(MLB_PATH)

print("Number of labels:", len(mlb.classes_))
print(mlb.classes_)

Number of labels: 20
['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']


In [10]:
def parse_labels(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return json.loads(value)

for df in [scotbess_df_train, scotbess_df_val, scotbess_df_test]:
    df["label_list"] = df["labels"].apply(parse_labels)


In [11]:
scotbess_y_train = mlb.transform(scotbess_df_train["label_list"])
scotbess_y_val = mlb.transform(scotbess_df_val["label_list"])
scotbess_y_test = mlb.transform(scotbess_df_test["label_list"])

In [12]:
scotbess_y_train.shape, scotbess_y_val.shape, scotbess_y_test.shape


((1340, 20), (165, 20), (170, 20))

In [13]:
scotbess_X_train = scotbess_df_train["final_masked_text"].fillna("").astype(str)
scotbess_X_val = scotbess_df_val["final_masked_text"].fillna("").astype(str)
scotbess_X_test = scotbess_df_test["final_masked_text"].fillna("").astype(str)

In [14]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [15]:
def tokenize(texts):
    return tokenizer(texts.tolist(), truncation=True, max_length=8192)
#dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

train_enc = tokenize(scotbess_X_train)
dev_enc = tokenize(scotbess_X_val)
test_enc = tokenize(scotbess_X_test)

In [16]:
y_train_bin = scotbess_y_train.astype(np.float32)
y_dev_bin = scotbess_y_val.astype(np.float32)
y_test_bin = scotbess_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)


['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']
(1340, 20)


In [17]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [18]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [19]:
#Learning curve nested subset selection
#multilabel stratification as in  Sechidis et al (2011)
def select_subset(parent_indices, labels, target_size, random_state=42):
    parent_indices = np.asarray(parent_indices)
    parent_labels = labels[parent_indices]

    splitter = MultilabelStratifiedShuffleSplit(n_splits=1, train_size=target_size,test_size = len(parent_indices) - target_size, random_state=random_state)

    dummy_X = np.zeros((len(parent_indices), 1))

    selected_positions, _ = next(splitter.split(dummy_X, parent_labels))

    # enforcing the exact requested size of the subset - this is a small tradeoff between exact subset size and optimal multilabel stratification
    #but differences were tiny when the size was not enforced;
    if len(selected_positions) < target_size:
        remaining_positions = np.setdiff1d(
            np.arange(len(parent_indices)),
            selected_positions)

        rng = np.random.default_rng(random_state)

        additional_positions = rng.choice(
            remaining_positions,
            size=target_size - len(selected_positions),
            replace=False)

        selected_positions = np.concatenate(
            [selected_positions, additional_positions])

    elif len(selected_positions) > target_size:
        rng = np.random.default_rng(random_state)

        selected_positions = rng.choice(
            selected_positions,
            size=target_size,
            replace=False )

    return parent_indices[selected_positions]

In [20]:
#saving one fixed set of nested subsets
#every smaller training subset is fully contained within every larger one

subset_indices_path = os.path.join(output_dir,"scotbess_learning_curve_indices.npz")

if os.path.exists(subset_indices_path):
    loaded = np.load(subset_indices_path)

    subsets = {int(size): loaded[size] for size in loaded.files}

    print("Loaded existing nested subsets.")

else:
    all_indices = np.arange(len(y_train_bin))
    subsets = {len(all_indices): all_indices.copy()}
    parent_indices = all_indices.copy()

    for target_size in  [1000, 750, 500, 250]:
        parent_indices = select_subset(
            parent_indices=parent_indices,
            labels=y_train_bin,
            target_size=target_size,
            random_state=42)
        subsets[target_size] = parent_indices.copy()

    np.savez(
        subset_indices_path,
        **{
            str(size): indices
            for size, indices in subsets.items()})

    print("Created and saved nested subsets.")

Loaded existing nested subsets.


In [21]:
training_sizes = [250, 500, 750, 1000]

for size in training_sizes:
    print(size, len(subsets[size]))

for smaller, larger in zip(training_sizes[:-1], training_sizes[1:]):
    nested = set(subsets[smaller]).issubset(set(subsets[larger]))
    print(f"{smaller} nested in {larger}: {nested}")
#subsets are correctly nested

250 250
500 500
750 750
1000 1000
250 nested in 500: True
500 nested in 750: True
750 nested in 1000: True


In [22]:
#label distributions
full_prevalence = y_train_bin.mean(axis=0)

coverage_rows = []

for size in training_sizes:
    subset_labels = y_train_bin[subsets[size]]
    subset_prevalence = subset_labels.mean(axis=0)

    coverage_rows.append({
        "training_size": size,
        "average_labels_per_document":
            subset_labels.sum(axis=1).mean(),
        "labels_with_zero_examples":
            int((subset_labels.sum(axis=0) == 0).sum()),
        "mean_absolute_prevalence_difference":
            np.abs(subset_prevalence - full_prevalence).mean()
    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(os.path.join(output_dir, "scotbess_learning_curve_coverage.csv"), index=False)
coverage_df

,training_size,average_labels_per_document,labels_with_zero_examples,mean_absolute_prevalence_difference
0,250,6.172000,0,0.009837
1,500,6.182000,0,0.010337
2,750,6.021333,0,0.002579
3,1000,6.030000,0,0.002737


In [23]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int) #default threshold

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {"f1_micro": f1_micro, "f1_macro": f1_macro}

In [24]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (1340, 20)
Val labels: (165, 20)
Test labels: (170, 20)
Number of labels: 20


**Training function**

In [25]:
#helpers for counting times for the final run with ModernBERT, as the run would get disconnected by colab  since it takes a long time to run it with 10 epochs on AAPD; reusing for SCOTBESS
class CheckpointTimeCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.time_log_path = os.path.join(output_dir, "time_log.json")
        self.previous_time_sec = 0.0
        self.session_start = None
        self.current_total_time_sec = 0.0
        self.current_session_time_sec = 0.0

    def on_train_begin(self, args, state, control, **kwargs):
        if os.path.exists(self.time_log_path):
            with open(self.time_log_path, "r") as f:
                self.previous_time_sec = json.load(f).get("train_time_sec", 0.0)
        else:
            self.previous_time_sec = 0.0

        self.session_start = time.perf_counter()
        self.current_total_time_sec = self.previous_time_sec
        self.current_session_time_sec = 0.0

    def _save_time(self, state):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self.current_session_time_sec = time.perf_counter() - self.session_start
        self.current_total_time_sec = self.previous_time_sec + self.current_session_time_sec

        data = {
            "train_time_sec": self.current_total_time_sec,
            "current_session_train_time_sec": self.current_session_time_sec,
            "previous_train_time_sec": self.previous_time_sec,
            "last_global_step": int(state.global_step),
            "last_epoch": float(state.epoch) if state.epoch is not None else None,
        }

        os.makedirs(self.output_dir, exist_ok=True)

        tmp_path = self.time_log_path + ".tmp"
        with open(tmp_path, "w") as f:
            json.dump(data, f, indent=2)

        os.replace(tmp_path, self.time_log_path)

    def on_save(self, args, state, control, **kwargs):
        self._save_time(state)

    def on_train_end(self, args, state, control, **kwargs):
        self._save_time(state)

    def get_times(self):
        if os.path.exists(self.time_log_path):
            with open(self.time_log_path, "r") as f:
                data = json.load(f)

            return (
                data.get("train_time_sec", self.current_total_time_sec),
                data.get("current_session_train_time_sec", self.current_session_time_sec),
            )

        return self.current_total_time_sec, self.current_session_time_sec

In [26]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, training_dataset, training_size, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)

    if measure_vram:
        reset_cuda_peak_memory()

    model = AutoModelForSequenceClassification.from_pretrained(
        config["base_model"],
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification",
        id2label=id2label,
        label2id=label2id,
        attn_implementation="sdpa")

    adapters.init(model)


    adapter_config = DoubleSeqBnConfig(reduction_factor=config["reduction_factor"])
    model.add_adapter("scotbess", config=adapter_config, set_active=True)
    model.train_adapter("scotbess")


    #Keep the task-specific ModernBERT classification head trainable
    for name, param in model.named_parameters():
       if name.startswith("head.") or name.startswith("classifier."):
            param.requires_grad = True

    #check whether the adapters are working
    print("Active adapters:", model.active_adapters)
    print(model.adapter_summary())
    #check the classification head
    print("Trainable classification/head parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad and ("head" in name.lower() or "classifier" in name.lower() or "classification" in name.lower()):
           print(name, param.numel())

    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["micro_batch_size"],
        per_device_eval_batch_size=config["micro_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],


        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none",
        label_names=["labels"],
        #gradient checkpointing
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False})

    time_callback = CheckpointTimeCallback(config["output_dir"])


    #adapter trainer, as recommended in the library's documentation
    trainer = AdapterTrainer(
        model=model,
        args=training_args,
        train_dataset=training_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"]), time_callback])

    #check
    print("Active adapters after trainer creation:", trainer.model.active_adapters)

    #for resuming if something goes wrong//collab's runtime gets disconnected
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


####imporved for the final run on MODERNBERT
    sync_cuda()
    # includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)
    sync_cuda()
    # accumulated training time saved after completed checkpoints/epochs
    train_time_sec, current_session_train_time_sec = time_callback.get_times()



    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "Scot-BESS",
        "method": "houlsby",
        "training_size": training_size,
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "micro_batch_size": config["micro_batch_size"],
        "gradient_accumulation_steps": config["gradient_accumulation_steps"],
        "effective_batch_size": config["effective_batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "current_session_train_time_sec": current_session_train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        #single forward pass, gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        #derive predictions, needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(config["output_dir"], f"classification_report_{training_size}_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(config["output_dir"], f"test_predictions_{training_size}_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")



    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

**Hyperparameter search**

In [27]:
#fixed params
learning_curve_config = {
    "base_model": "answerdotai/ModernBERT-base",
    "tokenizer_name": "answerdotai/ModernBERT-base",

    "max_length": 8192,
    "num_train_epochs": 30, #higher than for the original size
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 5, #higher than for the original size

    "reduction_factor": 8,

    #params from full-sized Scotbess Houlsby
    "learning_rate": 5e-4,
    "micro_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "effective_batch_size": 8,}


**Train and test**

In [28]:
training_seeds = [0, 1, 2]

learning_curve_output_dir = os.path.join(output_dir, "SCOTBESS_learning_curve_Houlsby")

os.makedirs(learning_curve_output_dir, exist_ok=True)

results_path = os.path.join(learning_curve_output_dir, "SCOTBESS_ModernBERT_Houlsby_learning_curve_results.csv")

if os.path.exists(results_path):
    existing_results = pd.read_csv(results_path)

    existing_results = (existing_results.drop_duplicates(subset=["training_size", "seed"],
            keep="last").sort_values(["training_size", "seed"]).reset_index(drop=True))

    learning_curve_results = existing_results.to_dict("records")

else:
    existing_results = pd.DataFrame()
    learning_curve_results = []

for training_size in training_sizes:
    subset_indices = subsets[training_size].tolist()
    training_subset = train_dataset.select(subset_indices)

    for seed in training_seeds:


        if not existing_results.empty:
            already_done = existing_results[(existing_results["training_size"] == training_size) & (existing_results["seed"] == seed)]

            if not already_done.empty:
                print(f"Skipping size={training_size}, seed={seed}")
                continue

        config = learning_curve_config.copy()
        config["output_dir"] = os.path.join(
            learning_curve_output_dir,
            f"size_{training_size}",
            f"seed_{seed}")

        result = run_training(
            config=config,
            training_dataset=training_subset,
            training_size=training_size,
            seed=seed,
            evaluate_test=True,
            measure_vram=True,
            save_report=True)

        learning_curve_results.append(result)

        pd.DataFrame(learning_curve_results).to_csv(results_path, index=False)

Skipping size=250, seed=0
Skipping size=250, seed=1
Skipping size=250, seed=2
Skipping size=500, seed=0
Skipping size=500, seed=1
Skipping size=500, seed=2
Skipping size=750, seed=0
Skipping size=750, seed=1
Skipping size=750, seed=2


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        6,526,080       4.378       1       1
--------------------------------------------------------------------------------
Full model                               149,064,960     100.000               0
Trainable classification/head parameters:
head.dense.weight 589824
head.norm.weight 768
classifier.weight 15360
classifier.bias 20
Active adapters after trainer creation: Stack[scotbess]
Resuming from checkpoint: /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_LC/SCOTBESS_MODERNBERT_HOULSBY_LC/SCOTBESS_learning_curve_Houlsby/size_1000/seed_0/checkpoint-375


...

Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        6,526,080       4.378       1       1
--------------------------------------------------------------------------------
Full model                               149,064,960     100.000               0
Trainable classification/head parameters:
head.dense.weight 589824
head.norm.weight 768
classifier.weight 15360
classifier.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,1.056400,0.447420,0.595183,0.458849
2,0.801100,0.348176,0.732515,0.598661
3,0.615200,0.292364,0.785908,0.682681
4,0.480000,0.269002,0.806436,0.775033
5,0.339600,0.247024,0.836901,0.800766
6,0.224200,0.288314,0.846078,0.823718
7,0.160400,0.271996,0.856996,0.836366
8,0.108700,0.283593,0.867265,0.840848
9,0.080100,0.301333,0.861239,0.837481
10,0.055000,0.301648,0.879597,0.865024


W0812 17:17:03.858000 1388 torch/_dynamo/convert_frame.py:1743] [4/8] torch._dynamo hit config.recompile_limit (8)
W0812 17:17:03.858000 1388 torch/_dynamo/convert_frame.py:1743] [4/8]    function: 'compiled_mlp' (/usr/local/lib/python3.12/dist-packages/transformers/models/modernbert/modeling_modernbert.py:528)
W0812 17:17:03.858000 1388 torch/_dynamo/convert_frame.py:1743] [4/8]    last reason: 4/7: 2 <= hidden_states.size()[0]  # return F.layer_norm(  # nn/modules/normalization.py:229 in forward (user code shown is first use of this value--the guard itself is not due user code but due to 0/1 specialization in the framework; to avoid specialization try torch._dynamo.decorators.mark_unbacked(tensor, dim))
W0812 17:17:03.858000 1388 torch/_dynamo/convert_frame.py:1743] [4/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0812 17:17:03.858000 1388 torch/_dynamo/convert_frame.py:1743] [4/8] To diagnose recompilation issues, see https://docs.pytorch.org/docs/main/user_guid

early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_LC/SCOTBESS_MODERNBERT_HOULSBY_LC/SCOTBESS_learning_curve_Houlsby/size_1000/seed_2/classification_report_1000_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_LC/SCOTBESS_MODERNBERT_HOULSBY_LC/SCOTBESS_learning_curve_Houlsby/size_1000/seed_2/test_predictions_1000_seed_2.npz


In [29]:
results_df = pd.read_csv(results_path)

learning_curve_summary = (
    results_df.groupby("training_size").agg(
        macro_f1_mean=("test_f1_macro", "mean"),
        macro_f1_std=("test_f1_macro", "std"),
        micro_f1_mean=("test_f1_micro", "mean"),
        micro_f1_std=("test_f1_micro", "std"),
        training_time_mean=("train_time_sec", "mean"),
        training_time_std=("train_time_sec", "std"),
        epochs_mean=("actual_epochs_trained", "mean")).reset_index())

learning_curve_summary.to_csv(os.path.join(output_dir, "SCOTBESS_Houlsby_learning_curve_summary.csv"), index=False)
learning_curve_summary

,training_size,macro_f1_mean,macro_f1_std,micro_f1_mean,micro_f1_std,training_time_mean,training_time_std,epochs_mean
0,250,0.733847,0.020939,0.792973,0.005303,2318.925415,488.019588,24.000000
1,500,0.823433,0.009846,0.852705,0.002284,4243.549182,605.002525,24.666667
2,750,0.850438,0.006878,0.871405,0.006436,5811.766763,1057.961276,22.333333
3,1000,0.862797,0.005127,0.880897,0.004480,6671.117861,470.326271,22.333333


In [30]:
from google.colab import runtime
runtime.unassign()